<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [0]:
#| echo: false
#| output: asis
show_doc(CrossChannelAttentionBlock)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/cross_channel_patchtst.py#L18){target="_blank" style="float:right; font-size:smaller"}

### CrossChannelAttentionBlock

```python
def CrossChannelAttentionBlock(
    d_model, n_heads, attn_dropout:float=0.0, dropout:float=0.0, bias:bool=True, pre_norm:bool=False
):
```

*Residual self-attention across channels at one time patch.*

In [0]:
#| echo: false
#| output: asis
show_doc(PatchTFTCrossChannel)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/cross_channel_patchtst.py#L56){target="_blank" style="float:right; font-size:smaller"}

### PatchTFTCrossChannel

```python
def PatchTFTCrossChannel(
    c_in, patch_size, patch_stride, num_patches, d_model, n_heads, d_ff, num_layers, augmentations:NoneType=None,
    mask_ratio:float=0.1, shared_embedding:bool=False, pretrain_head:bool=True, dropout:float=0.0,
    attn_dropout:float=0.0, act:str='gelu', pre_norm:bool=False, pe_type:str='tAPE', qkv_bias:bool=True,
    init_std:float=0.02, tokenizer_type:str='simple', tokenizer_kwargs:NoneType=None
):
```

*PatchTST with cross-channel attention after every temporal block.*

In [0]:
#| echo: false
#| output: asis
show_doc(SupervisedPatchTSTCrossChannel)

---

[source](https://github.com/benmfox/PhysioJEPA/blob/main/physiojepa/cross_channel_patchtst.py#L175){target="_blank" style="float:right; font-size:smaller"}

### SupervisedPatchTSTCrossChannel

```python
def SupervisedPatchTSTCrossChannel(
    c_in, patch_size, patch_stride, num_patches, d_model, n_heads, d_ff, num_layers, augmentations:NoneType=None,
    mask_ratio:float=0.0, shared_embedding:bool=False, dropout:float=0.0, attn_dropout:float=0.0, act:str='gelu',
    pre_norm:bool=False, pe_type:str='tAPE', qkv_bias:bool=True, init_std:float=0.02, tokenizer_type:str='simple',
    tokenizer_kwargs:NoneType=None, classifier_mlp_ratio:float=4.0, classifier_depth:int=1,
    classifier_init_std:float=0.02, classifier_qkv_bias:bool=True, classifier_complete_block:bool=True,
    classifier_affine:bool=False, num_classes:int=1
):
```

*End-to-end supervised PatchTST with cross-channel attention.*

In [ ]:
# Focused regression checks: parity when bypassed, channel mixing, and gradients.
torch.manual_seed(16)
encoder_kwargs = dict(
    c_in=3,
    patch_size=4,
    patch_stride=4,
    num_patches=8,
    d_model=16,
    n_heads=4,
    d_ff=32,
    num_layers=2,
    shared_embedding=False,
    pretrain_head=False,
    dropout=0.0,
    attn_dropout=0.0,
    pe_type='rotary',
    tokenizer_type='linear',
)
base = PatchTFTSimple(**encoder_kwargs).eval()
cross = PatchTFTCrossChannel(**encoder_kwargs).eval()
base_state = base.state_dict()
cross_base_state = {
    key: value
    for key, value in cross.state_dict().items()
    if not key.startswith('channel_layers.')
}
assert base_state.keys() == cross_base_state.keys()
assert all(base_state[key].shape == cross_base_state[key].shape for key in base_state)
incompatible = cross.load_state_dict(base_state, strict=False)
assert not incompatible.unexpected_keys
assert incompatible.missing_keys
assert all(key.startswith('channel_layers.') for key in incompatible.missing_keys)
cross.channel_layers = nn.ModuleList([nn.Identity() for _ in range(cross.num_layers)])
x = torch.randn(2, 3, 32)
with torch.no_grad():
    assert torch.allclose(base(x), cross(x), atol=1e-6, rtol=1e-5)

torch.manual_seed(16)
mixing_encoder = PatchTFTCrossChannel(**encoder_kwargs).eval()
changed_x = x.clone()
changed_x[:, 0] += 5.0
with torch.no_grad():
    unchanged_channel = mixing_encoder(x)[:, 1]
    mixed_channel = mixing_encoder(changed_x)[:, 1]
assert not torch.allclose(unchanged_channel, mixed_channel)

supervised_kwargs = dict(encoder_kwargs)
supervised_kwargs.pop('pretrain_head')
model = SupervisedPatchTSTCrossChannel(**supervised_kwargs, num_classes=1)
target = torch.tensor([[0.0], [1.0]])
logits = model(x)
assert logits.shape == (2, 1)
nn.BCEWithLogitsLoss()(logits, target).backward()
assert any(p.grad is not None for p in model.encoder.layers.parameters())
assert any(p.grad is not None for p in model.encoder.channel_layers.parameters())
assert any(p.grad is not None for p in model.classifier.parameters())